# Case Study: MovieLens Dataset SVD for Movie Recommendation

This notebook demonstrates how to build a basic movie recommendation system using the MovieLens dataset and Singular Value Decomposition (SVD). We will:
1. Load and preprocess the MovieLens dataset.
2. Create a user-item ratings matrix.
3. Apply SVD to decompose the matrix into latent features.
4. Reconstruct the ratings matrix and predict missing ratings.
5. Evaluate the model using RMSE and R2 scores.

Let's start by loading the movie and ratings data.


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error, r2_score
import math

### Load and Preprocess Movie Data

We will load the movie data from a URL and extract the relevant columns, such as `movie_id` and `movie_title`. The dataset will be merged with the ratings data to link the movie IDs to movie titles.


In [ ]:
# Define the URL to the MovieLens dataset (Movie IDs and Titles)
movies_url = "https://files.grouplens.org/datasets/movielens/ml-100k/u.item"

# Define column names to load the dataset correctly
movies_columns = ['movie_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL', 'unknown', 'action', 'adventure',
                  'animation', 'children', 'comedy', 'crime', 'documentary', 'drama', 'fantasy', 'film_noir', 'horror', 'musical',
                  'mystery', 'romance', 'sci_fi', 'thriller', 'war', 'western']

# Load the movie data into a pandas DataFrame, specifying the separator and encoding
movies_df = pd.read_csv(movies_url, sep='|', header=None, names=movies_columns, encoding='latin-1')

# Extract relevant columns: movie_id and movie_title
movies_df = movies_df[['movie_id', 'movie_title']]

# Display the first few rows of the extracted movie data
movies_df.head()


### Dataset Overview

The MovieLens 100k dataset consists of 100,000 ratings from 943 users on 1682 movies. Each rating is an integer between 1 and 5. We will focus on creating a user-item ratings matrix.
### Load and Preprocess Ratings Data

Now, we will load the ratings data from the MovieLens dataset and merge it with the movie data to associate movie titles with user ratings.



In [ ]:
# Load the MovieLens ratings data
ratings_url = "https://files.grouplens.org/datasets/movielens/ml-100k/u.data"
columns = ['user_id', 'movie_id', 'rating', 'timestamp']

# Load ratings data into a DataFrame
ratings_df = pd.read_csv(ratings_url, sep='\t', header=None, names=columns)

# Merge the ratings data with movie titles to link movie IDs to movie names
ratings_df = pd.merge(ratings_df, movies_df, on='movie_id')

# Create a user-item rating matrix where rows represent users and columns represent movies
ratings_matrix = ratings_df.pivot_table(index='user_id', columns='movie_title', values='rating')

# Fill missing values with 0 for now (assuming unrated movies are 0 for this demonstration)
ratings_matrix_filled = ratings_matrix.fillna(0)

# Display the first few rows of the ratings matrix
ratings_matrix_filled.head()


In [ ]:
ratings_matrix_filled.shape

### Apply Singular Value Decomposition (SVD)

We will now apply Singular Value Decomposition (SVD) to decompose the user-item ratings matrix into latent features. We will use these latent features to reconstruct the matrix and predict missing ratings.


## Explained variance ratio test to identify SVD Components

In [ ]:
# Let's run SVD for multiple values of n_components and check the explained variance ratio.
import matplotlib.pyplot as plt

# Compute explained variance for different n_components
explained_variance = []

for n in range(1, 301, 10):  # Testing components 1, 11, 21, ..., 301
    svd = TruncatedSVD(n_components=n)
    svd.fit(ratings_matrix_filled)
    explained_variance.append(np.sum(svd.explained_variance_ratio_))

# Plot the explained variance for different n_components
plt.figure(figsize=(10, 6))
plt.plot(range(1, 301, 10), explained_variance, marker='o')
plt.title('Explained Variance vs. Number of Components')
plt.xlabel('Number of Components')
plt.ylabel('Explained Variance')
plt.grid(True)
plt.show()


In [ ]:
# Perform Singular Value Decomposition (SVD) to decompose the ratings matrix into latent features
svd = TruncatedSVD(n_components=300)  # Using optimum number of components for better performance
latent_matrix = svd.fit_transform(ratings_matrix_filled)

# Reconstruct the original matrix using the decomposed matrices
reconstructed_matrix = np.dot(latent_matrix, svd.components_)

# Convert the reconstructed matrix back to a DataFrame
reconstructed_df = pd.DataFrame(reconstructed_matrix, columns=ratings_matrix.columns, index=ratings_matrix.index)

# Display the reconstructed matrix
reconstructed_df.head()


### Predict Missing Ratings

Next, we will predict the missing ratings for a specific user and movie. In this example, we will predict the rating for **User 1** and the movie **"Toy Story (1995)"**.


In [ ]:
# Predicting a missing rating for a specific user and movie
user_id = 1  # Example: User 1
movie_title = 'Toy Story (1995)'  # Example: Movie 'Toy Story (1995)'

# Check if the movie exists in the ratings matrix
if movie_title in reconstructed_df.columns:
    predicted_rating = reconstructed_df.loc[ratings_matrix.index[user_id-1], movie_title]
    print(f"\nPredicted Rating for User {user_id} and Movie '{movie_title}': {predicted_rating}")
else:
    print(f"\nMovie '{movie_title}' is not found in the ratings matrix!")


### Model Evaluation

We will evaluate the performance of the model by computing the **Root Mean Squared Error (RMSE)** and **R2 score** for the reconstructed ratings matrix compared to the original ratings matrix.


In [ ]:
# Compute RMSE (Root Mean Squared Error) for evaluating the reconstruction accuracy
original_flat = ratings_matrix_filled.values.flatten()
reconstructed_flat = reconstructed_matrix.flatten()

# Compute RMSE
rmse = math.sqrt(mean_squared_error(original_flat, reconstructed_flat))

# Compute R2 (coefficient of determination) score
r2 = r2_score(original_flat, reconstructed_flat)

# Display RMSE and R2
print(f"\nRMSE: {rmse}")
print(f"R2: {r2}")


### Summary

In this notebook, we:
1. Loaded and preprocessed the MovieLens dataset.
2. Created a user-item ratings matrix.
3. Applied Singular Value Decomposition (SVD) to decompose the matrix into latent features.
4. Predicted missing ratings using the reconstructed matrix.
5. Evaluated the model performance using RMSE and R2 scores.

The SVD-based recommendation system provides a basic yet powerful way to predict missing ratings and make personalized recommendations.
